In [2]:
import ast
import pandas as pd
import numpy as np
from scipy.special import softmax
from sklearn.metrics import roc_auc_score, average_precision_score


# DDI (wrt order)

|Order-->|2|3|4|4(unf)|5(unf)|6(unf)|7(unf)|
|---|---|---|---|---|---|---|---|
|N|500|500|20|500|500|500||
|Avg S (wrt gold)|2.11|2.07|4.64|0.07|0.59|||
|% S>0 (wrt gold)|0.618|0.596|0.75|0.524|0.55|||
|Acc (wrt all)|0.0055|0.0060|0.1|0.01|0.008|||
|AUROC (wrt all)|0.6263|0.6276|0.5824|0.6564|0.6434|||
|AUPRC (wrt all)|0.0055|0.0047|0.0678|0.0064|0.0051|||




unf : unfilitered, that is they are selected, without filtering for having their sublist in the record.

In [ ]:
order = 'three'
for model_name in ["Qwen-Qwen2.5-7B-Instruct", "Qwen-Qwen2.5-14B-Instruct", "Qwen-Qwen2.5-32B-Instruct"]:
  if order in ['four', 'five', 'six']:
    model_name_here = 'unfiltered_' + model_name
  else:
    model_name_here = model_name
    
  fdf = pd.read_json(f'../resultlogs/{model_name}/logger_logits_order_{order}_goldof{order}_againstall_{model_name_here}.jsonl', lines=True)
  fdf['S'] = fdf['yes_logit'] - fdf['no_logit']
  fdf.head()
  print(f"{model_name} | Order= {order}, N=", len(fdf[fdf.y == fdf.ygold]))
  print("Avg S (wrt gold)", np.mean(fdf[fdf.y == fdf.ygold].S))
  print("%S>0 (wrt gold)", sum(fdf[fdf.y == fdf.ygold].S > 0)/len(fdf[fdf.y == fdf.ygold]))

  fdf["p_candidate"] = (
      fdf.groupby("line_id")["S"]
        .transform(lambda x: softmax(x.to_numpy()))
  )

  fdf["label"] = (fdf["y"] == fdf["ygold"]).astype(int)

  auroc = roc_auc_score(fdf["label"], fdf["p_candidate"])
  auprc = average_precision_score(fdf["label"], fdf["p_candidate"])
  df_best = (
      fdf.loc[fdf.groupby("line_id")["p_candidate"].idxmax()]
        .reset_index(drop=True)
  )

  print(f"Accuracy: {len(df_best[df_best.y == df_best.ygold])/len(df_best):.4f}")

  print(f"AUROC: {auroc:.4f} | AUPRC: {auprc:.4f}")
  print("------")

Qwen-Qwen2.5-7B-Instruct | Order= three (unf), N= 500
Avg S (wrt gold) 2.071296875
%S>0 (wrt gold) 0.596
Accuracy: 0.0060
AUROC: 0.6276 | AUPRC: 0.0047
------
Qwen-Qwen2.5-14B-Instruct | Order= three (unf), N= 500
Avg S (wrt gold) -5.696
%S>0 (wrt gold) 0.328
Accuracy: 0.0060
AUROC: 0.6411 | AUPRC: 0.0048
------
Qwen-Qwen2.5-32B-Instruct | Order= three (unf), N= 500
Avg S (wrt gold) -0.382375
%S>0 (wrt gold) 0.518
Accuracy: 0.0080
AUROC: 0.6565 | AUPRC: 0.0052
------


In [114]:
fdf = pd.read_json('../resultlogs/logger_logits_order_five_goldoffive_againstall_unfiltered_Qwen-Qwen2.5-7B-Instruct.jsonl', lines=True)
fdf['S'] = fdf['yes_logit'] - fdf['no_logit']
fdf.head()
print("Order= 5 (unf), N=", len(fdf[fdf.y == fdf.ygold]))
print("Avg S (wrt gold)", np.mean(fdf[fdf.y == fdf.ygold].S))
print("%S>0 (wrt gold)", sum(fdf[fdf.y == fdf.ygold].S > 0)/len(fdf[fdf.y == fdf.ygold]))

fdf["p_candidate"] = (
    fdf.groupby("line_id")["S"]
      .transform(lambda x: softmax(x.to_numpy()))
)

fdf["label"] = (fdf["y"] == fdf["ygold"]).astype(int)

auroc = roc_auc_score(fdf["label"], fdf["p_candidate"])
auprc = average_precision_score(fdf["label"], fdf["p_candidate"])
df_best = (
    fdf.loc[fdf.groupby("line_id")["p_candidate"].idxmax()]
      .reset_index(drop=True)
)

print(f"Accuracy: {len(df_best[df_best.y == df_best.ygold])/len(df_best):.4f}")

print(f"AUROC: {auroc:.4f}")
print(f"AUPRC: {auprc:.4f}")

Order= 5 (unf), N= 500
Avg S (wrt gold) 0.593703125
%S>0 (wrt gold) 0.558
Accuracy: 0.0080
AUROC: 0.6434
AUPRC: 0.0051


In [71]:

fdf = pd.read_json('../resultlogs/logger_logits_order_four_goldoffour_againstall_Qwen-Qwen2.5-7B-Instruct.jsonl', lines=True)
fdf['S'] = fdf['yes_logit'] - fdf['no_logit']
fdf.head()
print("Order= 4, N=", len(fdf[fdf.y == fdf.ygold]))
print("Avg S (wrt gold)", np.mean(fdf[fdf.y == fdf.ygold].S))
print("%S>0 (wrt gold)", sum(fdf[fdf.y == fdf.ygold].S > 0)/len(fdf[fdf.y == fdf.ygold]))

fdf["p_candidate"] = (
    fdf.groupby("line_id")["S"]
      .transform(lambda x: softmax(x.to_numpy()))
)

fdf["label"] = (fdf["y"] == fdf["ygold"]).astype(int)

auroc = roc_auc_score(fdf["label"], fdf["p_candidate"])
auprc = average_precision_score(fdf["label"], fdf["p_candidate"])
df_best = (
    fdf.loc[fdf.groupby("line_id")["p_candidate"].idxmax()]
      .reset_index(drop=True)
)

print(f"Accuracy: {len(df_best[df_best.y == df_best.ygold])/len(df_best):.4f}")

print(f"AUROC: {auroc:.4f}")
print(f"AUPRC: {auprc:.4f}")

Order= 4, N= 20
Avg S (wrt gold) 4.64609375
%S>0 (wrt gold) 0.75
Accuracy: 0.1000
AUROC: 0.5824
AUPRC: 0.0678


In [83]:
fdf = pd.read_json('../resultlogs/logger_logits_order_four_goldoffour_againstall_unfiltered_Qwen-Qwen2.5-7B-Instruct.jsonl', lines=True)
fdf['S'] = fdf['yes_logit'] - fdf['no_logit']
fdf.head()
print("Order= 4 (unf), N=", len(fdf[fdf.y == fdf.ygold]))
print("Avg S (wrt gold)", np.mean(fdf[fdf.y == fdf.ygold].S))
print("%S>0 (wrt gold)", sum(fdf[fdf.y == fdf.ygold].S > 0)/len(fdf[fdf.y == fdf.ygold]))

fdf["p_candidate"] = (
    fdf.groupby("line_id")["S"]
      .transform(lambda x: softmax(x.to_numpy()))
)

fdf["label"] = (fdf["y"] == fdf["ygold"]).astype(int)

auroc = roc_auc_score(fdf["label"], fdf["p_candidate"])
auprc = average_precision_score(fdf["label"], fdf["p_candidate"])
df_best = (
    fdf.loc[fdf.groupby("line_id")["p_candidate"].idxmax()]
      .reset_index(drop=True)
)

print(f"Accuracy: {len(df_best[df_best.y == df_best.ygold])/len(df_best):.4f}")

print(f"AUROC: {auroc:.4f}")
print(f"AUPRC: {auprc:.4f}")

Order= 4 (unf), N= 500
Avg S (wrt gold) 0.071625
%S>0 (wrt gold) 0.524
Accuracy: 0.0100
AUROC: 0.6564
AUPRC: 0.0064


In [133]:
fdf = pd.read_json('../resultlogs/logger_logits_order_four_goldoffour_againstall_unfiltered_Echelon-AI-Med-Qwen2-7B.jsonl', lines=True)
fdf['S'] = fdf['yes_logit'] - fdf['no_logit']
fdf.head()
print("Echelon-AI-Med-Qwen2-7B | Order= 4 (unf), N=", len(fdf[fdf.y == fdf.ygold]))
print("Avg S (wrt gold)", np.mean(fdf[fdf.y == fdf.ygold].S))
print("%S>0 (wrt gold)", sum(fdf[fdf.y == fdf.ygold].S > 0)/len(fdf[fdf.y == fdf.ygold]))

fdf["p_candidate"] = (
    fdf.groupby("line_id")["S"]
      .transform(lambda x: softmax(x.to_numpy()))
)

fdf["label"] = (fdf["y"] == fdf["ygold"]).astype(int)

auroc = roc_auc_score(fdf["label"], fdf["p_candidate"])
auprc = average_precision_score(fdf["label"], fdf["p_candidate"])
df_best = (
    fdf.loc[fdf.groupby("line_id")["p_candidate"].idxmax()]
      .reset_index(drop=True)
)

print(f"Accuracy: {len(df_best[df_best.y == df_best.ygold])/len(df_best):.4f}")

print(f"AUROC: {auroc:.4f}")
print(f"AUPRC: {auprc:.4f}")

Echelon-AI-Med-Qwen2-7B | Order= 4 (unf), N= 500
Avg S (wrt gold) 0.6665859375
%S>0 (wrt gold) 0.734
Accuracy: 0.0080
AUROC: 0.6534
AUPRC: 0.0064


In [126]:
fdf = pd.read_json('../resultlogs/logger_logits_order_three_goldofthree_againstall_Qwen-Qwen2.5-7B-Instruct.jsonl', lines=True)
fdf['S'] = fdf['yes_logit'] - fdf['no_logit']
fdf.head()
print("Qwen-Qwen2.5-7B-Instruct | Order= 3, N=", len(fdf[fdf.y == fdf.ygold]))
print("Avg S (wrt gold)", np.mean(fdf[fdf.y == fdf.ygold].S))
print("%S>0 (wrt gold)", sum(fdf[fdf.y == fdf.ygold].S > 0)/len(fdf[fdf.y == fdf.ygold]))

fdf["p_candidate"] = (
    fdf.groupby("line_id")["S"]
      .transform(lambda x: softmax(x.to_numpy()))
)

fdf["label"] = (fdf["y"] == fdf["ygold"]).astype(int)

auroc = roc_auc_score(fdf["label"], fdf["p_candidate"])
auprc = average_precision_score(fdf["label"], fdf["p_candidate"])
df_best = (
    fdf.loc[fdf.groupby("line_id")["p_candidate"].idxmax()]
      .reset_index(drop=True)
)

print(f"Accuracy: {len(df_best[df_best.y == df_best.ygold])/len(df_best):.4f}")

print(f"AUROC: {auroc:.4f}")
print(f"AUPRC: {auprc:.4f}")

Qwen-Qwen2.5-7B-Instruct | Order= 3, N= 500
Avg S (wrt gold) 2.071296875
%S>0 (wrt gold) 0.596
Accuracy: 0.0060
AUROC: 0.6276
AUPRC: 0.0047


In [131]:
fdf = pd.read_json('../resultlogs/logger_logits_order_three_goldofthree_againstall_Echelon-AI-Med-Qwen2-7B.jsonl', lines=True)
fdf['S'] = fdf['yes_logit'] - fdf['no_logit']
fdf.head()
print("Echelon-AI-Med-Qwen2-7B | Order= 3, N=", len(fdf[fdf.y == fdf.ygold]))
print("Avg S (wrt gold)", np.mean(fdf[fdf.y == fdf.ygold].S))
print("%S>0 (wrt gold)", sum(fdf[fdf.y == fdf.ygold].S > 0)/len(fdf[fdf.y == fdf.ygold]))

fdf["p_candidate"] = (
    fdf.groupby("line_id")["S"]
      .transform(lambda x: softmax(x.to_numpy()))
)

fdf["label"] = (fdf["y"] == fdf["ygold"]).astype(int)

auroc = roc_auc_score(fdf["label"], fdf["p_candidate"])
auprc = average_precision_score(fdf["label"], fdf["p_candidate"])
df_best = (
    fdf.loc[fdf.groupby("line_id")["p_candidate"].idxmax()]
      .reset_index(drop=True)
)

print(f"Accuracy: {len(df_best[df_best.y == df_best.ygold])/len(df_best):.4f}")

print(f"AUROC: {auroc:.4f}")
print(f"AUPRC: {auprc:.4f}")

Echelon-AI-Med-Qwen2-7B | Order= 3, N= 500
Avg S (wrt gold) 0.8045625
%S>0 (wrt gold) 0.774
Accuracy: 0.0040
AUROC: 0.6239
AUPRC: 0.0047


In [134]:
fdf = pd.read_json('../resultlogs/logger_logits_order_two_goldoftwo_againstall_Echelon-AI-Med-Qwen2-7B.jsonl', lines=True)
fdf['S'] = fdf['yes_logit'] - fdf['no_logit']
fdf.head()
print("Echelon-AI-Med-Qwen2-7B | Order= 2, N=", len(fdf[fdf.y == fdf.ygold]))
print("Avg S (wrt gold)", np.mean(fdf[fdf.y == fdf.ygold].S))
print("%S>0 (wrt gold)", sum(fdf[fdf.y == fdf.ygold].S > 0)/len(fdf[fdf.y == fdf.ygold]))

fdf["p_candidate"] = (
    fdf.groupby("line_id")["S"]
      .transform(lambda x: softmax(x.to_numpy()))
)

fdf["label"] = (fdf["y"] == fdf["ygold"]).astype(int)

auroc = roc_auc_score(fdf["label"], fdf["p_candidate"])
auprc = average_precision_score(fdf["label"], fdf["p_candidate"])
df_best = (
    fdf.loc[fdf.groupby("line_id")["p_candidate"].idxmax()]
      .reset_index(drop=True)
)

print(f"Accuracy: {len(df_best[df_best.y == df_best.ygold])/len(df_best):.4f}")

print(f"AUROC: {auroc:.4f}")
print(f"AUPRC: {auprc:.4f}")

Echelon-AI-Med-Qwen2-7B | Order= 2, N= 500
Avg S (wrt gold) 0.6618984375
%S>0 (wrt gold) 0.74
Accuracy: 0.0160
AUROC: 0.6326
AUPRC: 0.0057


In [69]:
fdf = pd.read_json('../resultlogs/logger_logits_order_two_goldoftwo_againstall_Qwen-Qwen2.5-7B-Instruct.jsonl', lines=True)
fdf['S'] = fdf['yes_logit'] - fdf['no_logit']
fdf.head()
print("Order= 2, N=", len(fdf[fdf.y == fdf.ygold]))
print("Avg S (wrt gold)", np.mean(fdf[fdf.y == fdf.ygold].S))
print("%S>0 (wrt gold)", sum(fdf[fdf.y == fdf.ygold].S > 0)/len(fdf[fdf.y == fdf.ygold]))

fdf["p_candidate"] = (
    fdf.groupby("line_id")["S"]
      .transform(lambda x: softmax(x.to_numpy()))
)

fdf["label"] = (fdf["y"] == fdf["ygold"]).astype(int)

auroc = roc_auc_score(fdf["label"], fdf["p_candidate"])
auprc = average_precision_score(fdf["label"], fdf["p_candidate"])

df_best = (
    fdf.loc[fdf.groupby("line_id")["p_candidate"].idxmax()]
      .reset_index(drop=True)
)

print(f"Accuracy: {len(df_best[df_best.y == df_best.ygold])/len(df_best):.4f}")
print(f"AUROC: {auroc:.4f}")
print(f"AUPRC: {auprc:.4f}")


Order= 2, N= 500
Avg S (wrt gold) 2.112953125
%S>0 (wrt gold) 0.618
Accuracy: 0.0160
AUROC: 0.6263
AUPRC: 0.0055


In [79]:
df = pd.read_csv('../../HODDI/dataset/HODDI_v1/dictionary/Drugbank_ID_SMILE_all_structure links.csv')
drugid2drugname = dict(zip(df["DrugBank ID"].values, df["Name"].values ))

maindf = pd.read_csv("../../HODDI/dataset/HODDI_v1/HODDI/Merged_Dataset/pos.csv")
maindf["DrugBankID"] = maindf["DrugBankID"].apply(ast.literal_eval)
maindf["DrugBankID_sorted"] = [sorted(dbl) for dbl in maindf["DrugBankID"].values]
drug_lists = maindf["DrugBankID"].values
side_effect_list = maindf["SE_above_0.9"].values
maindf["len"] = [len(dl) for dl in maindf.DrugBankID_sorted.values]
maindf["slen"] = [len(set(dl)) for dl in maindf.DrugBankID_sorted.values]
maindf.head()


ORDER = 10
subdf = maindf[(maindf.len == ORDER) & (maindf.slen == ORDER)]
subdf

mask = []
for dl in subdf.DrugBankID_sorted.values:
    flag = 1
    for d in dl:
        if d not in drugid2drugname:
            flag = 0
            break
    if flag == 1:
        mask.append(True)
    else:
        mask.append(False)

print(len(subdf), len(subdf[mask]), len(set(subdf[mask]["SE_above_0.9"].values)))

fp = open(f"../data/{ORDER}-list-unfiltered.txt", "w")
for i in subdf[mask].index.values:
    print(i, file=fp)
fp.close()

subdf[mask]

2588 2123 908


,report_id,SE_above_0.9,DrugBankID,hyperedge_label,time,row_index,DrugBankID_sorted,len,slen
25,12965424,C0919671,"[DB00313, DB00502, DB09287, DB01914, DB01390, ...",1,2016Q4,26,"[DB00213, DB00313, DB00363, DB00502, DB00612, ...",10,10
124,12510191,C1527311,"[DB01175, DB01129, DB01222, DB00983, DB06605, ...",1,2016Q4,125,"[DB00277, DB00471, DB00695, DB00966, DB00983, ...",10,10
178,11584681,C0085580,"[DB00387, DB00230, DB00734, DB00829, DB00695, ...",1,2015Q4,179,"[DB00186, DB00230, DB00313, DB00370, DB00387, ...",10,10
179,14381475,C0426732,"[DB00678, DB00451, DB00335, DB06724, DB11094, ...",1,2021Q2,180,"[DB00136, DB00331, DB00335, DB00381, DB00451, ...",10,10
201,13005738,C0085639,"[DB00879, DB00625, DB14126, DB00999, DB01050, ...",1,2016Q4,202,"[DB00245, DB00334, DB00381, DB00502, DB00625, ...",10,10
...,...,...,...,...,...,...,...,...,...
110742,19491099,C3160741,"[DB00434, DB00653, DB01156, DB00327, DB00202, ...",1,2021Q3,110743,"[DB00186, DB00202, DB00292, DB00327, DB00434, ...",10,10
110753,9835566,C0018418,"[DB00331, DB01261, DB00404, DB01104, DB00999, ...",1,2015Q1,110754,"[DB00264, DB00331, DB00381, DB00384, DB00404, ...",10,10
110756,15777822,C0015397,"[DB14513, DB00722, DB00540, DB00588, DB11075, ...",1,2018Q4,110757,"[DB00476, DB00540, DB00588, DB00654, DB00722, ...",10,10
110864,12537120,C0575081,"[DB00196, DB01015, DB00440, DB06218, DB00555, ...",1,2016Q3,110865,"[DB00196, DB00300, DB00440, DB00503, DB00555, ...",10,10


# lower associates

|agg.|metric|2|3|4|5|
|---|---|---|---|---|---|
|max S|Avg S (wrt gold)|||||
||% S>0 (wrt gold)|||||
|min S|Avg S (wrt gold)|||||
||% S>0 (wrt gold)|||||
|mean S|Avg S (wrt gold)|||||
||% S>0 (wrt gold)|||||

In [87]:
ls

results_analysis.ipynb  slurm-general/


In [107]:
fdf = pd.read_json('../resultlogs/logger_logits_order_two_goldoftwo_againstall_Qwen-Qwen2.5-7B-Instruct.jsonl', lines=True)
fdf['S'] = fdf['yes_logit'] - fdf['no_logit']
gfdf = fdf[fdf.y == fdf.ygold]

ldf = pd.read_json("../resultlogs/logger_logits_order_two_goldoftwo_given_one_Qwen-Qwen2.5-7B-Instruct.jsonl", lines=True)
ldf['S'] = ldf['yes_logit'] - ldf['no_logit']

aggldf = ldf.loc[ldf.groupby("line_id")["S"].idxmax()].reset_index(drop=True)
print("Order= 2, N=", len(aggldf))
print("Max S =========")
print("Avg S (wrt gold)", np.mean(aggldf.S))
print("%S>0 (wrt gold)", sum(aggldf.S > 0)/len(aggldf))
print("Avg M (wrt gold)", np.mean(gfdf.S - aggldf.S))

aggldf = ldf.loc[ldf.groupby("line_id")["S"].idxmin()].reset_index(drop=True)
print("Min S =========")
print("Avg S (wrt gold)", np.mean(aggldf.S))
print("%S>0 (wrt gold)", sum(aggldf.S > 0)/len(aggldf))
print("Avg M (wrt gold)", np.mean(gfdf.S - aggldf.S))

aggldf = ldf.groupby("line_id")[["yes_logit", "no_logit","S"]].mean()
print("Mean S =========")
print("Avg S (wrt gold)", np.mean(aggldf.S))
print("%S>0 (wrt gold)", sum(aggldf.S > 0)/len(aggldf))
print("Avg M (wrt gold)", np.mean(gfdf.S - aggldf.S))


Order= 2, N= 500
Max S =========
Avg S (wrt gold) 7.915078125
%S>0 (wrt gold) 0.786
Avg M (wrt gold) -11.34375
Min S =========
Avg S (wrt gold) 1.424109375
%S>0 (wrt gold) 0.526
Avg M (wrt gold) -1.46875
Mean S =========
Avg S (wrt gold) 4.66959375
%S>0 (wrt gold) 0.688
Avg M (wrt gold) -6.40625


In [109]:
fdf = pd.read_json('../resultlogs/logger_logits_order_three_goldofthree_againstall_Qwen-Qwen2.5-7B-Instruct.jsonl', lines=True)
fdf['S'] = fdf['yes_logit'] - fdf['no_logit']
gfdf = fdf[fdf.y == fdf.ygold]

ldf = pd.read_json("../resultlogs/logger_logits_order_three_goldofthree_given_two_Qwen-Qwen2.5-7B-Instruct.jsonl", lines=True)
ldf['S'] = ldf['yes_logit'] - ldf['no_logit']

aggldf = ldf.loc[ldf.groupby("line_id")["S"].idxmax()].reset_index(drop=True)
print("Order= 3, N=", len(aggldf))
print("Max S =========")
print("Avg S (wrt gold)", np.mean(aggldf.S))
print("%S>0 (wrt gold)", sum(aggldf.S > 0)/len(aggldf))
print("Avg M (wrt gold)", np.mean(gfdf.S - aggldf.S))

aggldf = ldf.loc[ldf.groupby("line_id")["S"].idxmin()].reset_index(drop=True)
print("Min S =========")
print("Avg S (wrt gold)", np.mean(aggldf.S))
print("%S>0 (wrt gold)", sum(aggldf.S > 0)/len(aggldf))
print("Avg M (wrt gold)", np.mean(gfdf.S - aggldf.S))

aggldf = ldf.groupby("line_id")[["yes_logit", "no_logit","S"]].mean()
print("Mean S =========")
print("Avg S (wrt gold)", np.mean(aggldf.S))
print("%S>0 (wrt gold)", sum(aggldf.S > 0)/len(aggldf))
print("Avg M (wrt gold)", np.mean(gfdf.S - aggldf.S))

Order= 3, N= 500
Max S =========
Avg S (wrt gold) 4.1894375
%S>0 (wrt gold) 0.692
Avg M (wrt gold) -0.3046875
Min S =========
Avg S (wrt gold) -1.78496875
%S>0 (wrt gold) 0.434
Avg M (wrt gold) 3.515625
Mean S =========
Avg S (wrt gold) 1.1553125
%S>0 (wrt gold) 0.572
Avg M (wrt gold) 2.153645833333333


In [111]:
fdf = pd.read_json('../resultlogs/logger_logits_order_four_goldoffour_againstall_unfiltered_Qwen-Qwen2.5-7B-Instruct.jsonl', lines=True)
fdf['S'] = fdf['yes_logit'] - fdf['no_logit']
gfdf = fdf[fdf.y == fdf.ygold]

ldf = pd.read_json("../resultlogs/logger_logits_order_four_goldoffour_given_three_unfiltered_Qwen-Qwen2.5-7B-Instruct.jsonl", lines=True)
ldf['S'] = ldf['yes_logit'] - ldf['no_logit']

aggldf = ldf.loc[ldf.groupby("line_id")["S"].idxmax()].reset_index(drop=True)
print("Order= 3, N=", len(aggldf))
print("Max S =========")
print("Avg S (wrt gold)", np.mean(aggldf.S))
print("%S>0 (wrt gold)", sum(aggldf.S > 0)/len(aggldf))
print("Avg M (wrt gold)", np.mean(gfdf.S - aggldf.S))

aggldf = ldf.loc[ldf.groupby("line_id")["S"].idxmin()].reset_index(drop=True)
print("Min S =========")
print("Avg S (wrt gold)", np.mean(aggldf.S))
print("%S>0 (wrt gold)", sum(aggldf.S > 0)/len(aggldf))
print("Avg M (wrt gold)", np.mean(gfdf.S - aggldf.S))

aggldf = ldf.groupby("line_id")[["yes_logit", "no_logit","S"]].mean()
print("Mean S =========")
print("Avg S (wrt gold)", np.mean(aggldf.S))
print("%S>0 (wrt gold)", sum(aggldf.S > 0)/len(aggldf))
print("Avg M (wrt gold)", np.mean(gfdf.S - aggldf.S))

Order= 3, N= 500
Max S =========
Avg S (wrt gold) 2.675953125
%S>0 (wrt gold) 0.658
Avg M (wrt gold) -18.96875
Min S =========
Avg S (wrt gold) -3.467734375
%S>0 (wrt gold) 0.346
Avg M (wrt gold) -14.8203125
Mean S =========
Avg S (wrt gold) -0.26303515625
%S>0 (wrt gold) 0.514
Avg M (wrt gold) -16.953125


In [118]:
fdf = pd.read_json('../resultlogs/logger_logits_order_two_goldoftwo_againstall_Qwen-Qwen2.5-7B-Instruct.jsonl', lines=True)
fdf['S'] = fdf['yes_logit'] - fdf['no_logit']
gfdf = fdf[fdf.y == fdf.ygold]

ldf_w_info = pd.read_json("../resultlogs/logger_logits_order_two_goldoftwo_given_one_w_info_Qwen-Qwen2.5-7B-Instruct.jsonl", lines= True)
ldf_w_info['S'] = ldf_w_info['yes_logit'] - ldf_w_info['no_logit']
ldf_w_info


print("Order= 2, N=", len(ldf_w_info))
print("Max S =========")
print("Avg S (wrt gold)", np.mean(ldf_w_info.S))
print("%S>0 (wrt gold)", sum(ldf_w_info.S > 0)/len(ldf_w_info))
print("Avg M (wrt gold)", np.mean(gfdf.S - ldf_w_info.S))

Order= 2, N= 500
Max S =========
Avg S (wrt gold) -7.632265625
%S>0 (wrt gold) 0.19
Avg M (wrt gold) 12.78125


In [119]:
fdf = pd.read_json('../resultlogs/logger_logits_order_three_goldofthree_againstall_Qwen-Qwen2.5-7B-Instruct.jsonl', lines=True)
fdf['S'] = fdf['yes_logit'] - fdf['no_logit']
gfdf = fdf[fdf.y == fdf.ygold]

ldf_w_info = pd.read_json("../resultlogs/logger_logits_order_three_goldofthree_given_two_w_info_Qwen-Qwen2.5-7B-Instruct.jsonl", lines= True)
ldf_w_info['S'] = ldf_w_info['yes_logit'] - ldf_w_info['no_logit']
ldf_w_info


print("Order= 3, N=", len(ldf_w_info))
print("Max S =========")
print("Avg S (wrt gold)", np.mean(ldf_w_info.S))
print("%S>0 (wrt gold)", sum(ldf_w_info.S > 0)/len(ldf_w_info))
print("Avg M (wrt gold)", np.mean(gfdf.S - ldf_w_info.S))

Order= 3, N= 500
Max S =========
Avg S (wrt gold) -7.460328125
%S>0 (wrt gold) 0.176
Avg M (wrt gold) 11.40625


In [ ]:
fdf = pd.read_json('../resultlogs/logger_logits_order_four_goldoffour_againstall_unfiltered_Qwen-Qwen2.5-7B-Instruct.jsonl', lines=True)
fdf['S'] = fdf['yes_logit'] - fdf['no_logit']
gfdf = fdf[fdf.y == fdf.ygold]

ldf_w_info = pd.read_json("../resultlogs/logger_logits_order_four_goldoffour_given_three_w_info_Qwen-Qwen2.5-7B-Instruct.jsonl", lines= True)
ldf_w_info['S'] = ldf_w_info['yes_logit'] - ldf_w_info['no_logit']
ldf_w_info


print("Order= 3, N=", len(ldf_w_info))
print("Max S =========")
print("Avg S (wrt gold)", np.mean(ldf_w_info.S))
print("%S>0 (wrt gold)", sum(ldf_w_info.S > 0)/len(ldf_w_info))
print("Avg M (wrt gold)", np.mean(gfdf.S - ldf_w_info.S))